# Deutsch–Jozsa classification

Distinguish a balanced oracle from a constant oracle using exact output probabilities.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [ ]:
def deutsch_jozsa(balanced):
    n = 4
    circuit = QuantumCircuit(n + 1)
    circuit.x(n)
    circuit.h(range(n + 1))
    if balanced:
        for wire in range(n):
            circuit.cx(wire, n)
    circuit.h(range(n))
    return circuit

circuits = [deutsch_jozsa(False), deutsch_jozsa(True)]

def reference_probabilities():
    return np.asarray([Statevector.from_instruction(c).probabilities(qargs=range(4)) for c in circuits])

reference, reference_ms, _ = benchmark(reference_probabilities)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]

def mettleq_probabilities():
    values = []
    for circuit in compiled:
        state = backend.run(circuit, shots=1, return_statevector=True).result().data(0)["statevector"]
        values.append(Statevector(state).probabilities(qargs=range(4)))
    return np.asarray(values)

candidate, mettleq_ms, _ = benchmark(mettleq_probabilities)
error = max_abs_error(reference, candidate)
classes = [int(np.argmax(row) != 0) for row in candidate]
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/05_deutsch_jozsa.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="probability vector atol=2e-6 and exact oracle class",
    passed=error <= 2e-6 and classes == [0, 1],
    exact_match=classes == [0, 1],
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "classifications": classes},
)